# 🧬 Panacée — Phase 3 : Multi-propriétés + IA (Kaggle)

**Phase 3** : Efficacité anti-VIH, solubilité, LogP, BBB, métabolisme, synergie (IA Raisonnement).  
Charge l'encodeur Phase 2 → produit `panacee_phase3_complete.pth`.

### Chaîne complète :
```
Phase 1 → sovereign_encoder_v1.pth
         ↓
Phase 2 → best_toxicity_model.pth    ← INPUT requis ici
         ↓
Phase 3 (ce notebook) → panacee_phase3_complete.pth
```

### Avant de lancer :
1. Dashboard local + ngrok actifs
2. **Ajouter `best_toxicity_model.pth` en Input Kaggle** :
   - Depuis Kaggle Dataset : + Add Data → votre dataset panacee-phase2
   - Ou uploader le fichier manuellement
3. Mettre à jour `PHASE2_CKPT` avec le chemin correct
4. ▶▶ Run All

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CONFIGURATION — modifier ces lignes ONLY    ║
# ╚══════════════════════════════════════════════╝

NGROK_URL  = "https://unintimidated-feelingly-reva.ngrok-free.dev"          # URL publique de votre dashboard
TOKEN      = "panacee_kaggle_2026"                       # == PANACEE_INGEST_TOKEN dans .env
RUN_NAME   = "kaggle_phase3_run01"                   # Nom affiché dans le dashboard
N_EPOCHS   = 300  # GPU: 30-80, CPU: 10-20

# Checkpoint Phase 2 (REQUIS)
# Chemin dans /kaggle/input/<nom_du_dataset>/<fichier>.pth
PHASE2_CKPT = "/kaggle/input/panacee-phase2/best_toxicity_model.pth"

MAX_MOLECULES = None

print(f"Run         : {RUN_NAME}")
print(f"Epochs      : {N_EPOCHS}")
print(f"Phase 2     : {PHASE2_CKPT}")

In [ ]:
# ── Vérifications préalables ────────────────────────────────────────────────
import json
import urllib.request
from pathlib import Path

# 1) Dashboard
try:
    r = urllib.request.urlopen(NGROK_URL.rstrip('/') + '/api/health', timeout=8)
    print(f"✅ Dashboard joignable : {json.loads(r.read())}")
except Exception as e:
    print(f"❌ Dashboard NON joignable : {e}")
    raise SystemExit("Corrigez NGROK_URL avant de continuer.")

# 2) Checkpoint Phase 2
if not Path(PHASE2_CKPT).exists():
    print(f"❌ Checkpoint Phase 2 introuvable : {PHASE2_CKPT}")
    print()
    print("Pour l'ajouter en Input Kaggle :")
    print("  Option A) + Add Data → votre dataset panacee-phase2")
    print("  Option B) Uploader best_toxicity_model.pth manuellement")
    print("  Puis mettre à jour PHASE2_CKPT avec le chemin /kaggle/input/...")
    raise SystemExit("Checkpoint Phase 2 manquant.")

size_mb = Path(PHASE2_CKPT).stat().st_size / 1e6
print(f"✅ Checkpoint Phase 2 : {PHASE2_CKPT} ({size_mb:.1f} MB)")

In [ ]:
# ── Variables d'environnement ────────────────────────────────────────────────
import os

os.environ["PANACEE_PUSH_URL"]   = NGROK_URL
os.environ["PANACEE_PUSH_TOKEN"] = TOKEN
os.environ["PANACEE_PUSH_RUN"]   = RUN_NAME
os.environ["PYTHONIOENCODING"]   = "utf-8"

print("Variables ✓")

In [ ]:
# ── Dépendances ──────────────────────────────────────────────────────────────
import subprocess
import sys


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch-geometric", "rdkit", "deepchem")
print("Dépendances ✓")

In [ ]:
# ── Cloner le repo ──────────────────────────────────────────────────────────
CLONE_DIR = Path("/kaggle/working/panacee")
REPO_URL  = "https://github.com/jumoras0000/savh_gnn.git"

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)

PROJET = CLONE_DIR / "Projet_Panacee"
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))
print(f"Projet : {PROJET}")

In [ ]:
# ── Rendre le checkpoint Phase 2 accessible au projet ──────────────────────
# run_phase3.py cherche par défaut checkpoints/phase2/best_toxicity_model.pth
# On copie le fichier d'Input à cet emplacement.
import shutil

PHASE2_INTERNAL = "/kaggle/working/checkpoints/phase2"
Path(PHASE2_INTERNAL).mkdir(parents=True, exist_ok=True)
shutil.copy(PHASE2_CKPT, f"{PHASE2_INTERNAL}/best_toxicity_model.pth")
print(f"✅ Phase 2 copié → {PHASE2_INTERNAL}/best_toxicity_model.pth")

SAVE_DIR = "/kaggle/working/checkpoints/phase3"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
print(f"Phase 3 checkpoints → {SAVE_DIR}")

In [ ]:
# ── Lancer Phase 3 ──────────────────────────────────────────────────────────
import os as _os

_os.chdir(str(PROJET))

# PANACEE_CKPT_ROOT → le projet trouvera phase2/best_toxicity_model.pth
os.environ["PANACEE_CKPT_ROOT"] = "/kaggle/working/checkpoints"

sys.argv = [
    "run_phase3.py",
    "--download",
    "--pretrained_model", f"{PHASE2_INTERNAL}/best_toxicity_model.pth",
    "--save_dir",         SAVE_DIR,
    "--epochs",           str(N_EPOCHS),
    "--skip_checks",      # pré-requis vérifiés dans les cellules ci-dessus
]

print(f"Phase 3 → run={RUN_NAME}, save={SAVE_DIR}")
print(f"Push  → {os.environ['PANACEE_PUSH_URL']}")
print()

from run_phase3 import main

main()

In [ ]:
# ── Export du modèle complet ────────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/output/panacee-phase3"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

ckpt_src = f"{SAVE_DIR}/panacee_phase3_complete.pth"
if Path(ckpt_src).exists():
    shutil.copy(ckpt_src, f"{OUTPUT_DIR}/panacee_phase3_complete.pth")
    size_mb = Path(ckpt_src).stat().st_size / 1e6
    print(f"✅ Modèle complet exporté ({size_mb:.1f} MB)")
else:
    # Chercher le meilleur checkpoint disponible
    candidates = list(Path(SAVE_DIR).glob("*.pth"))
    if candidates:
        best = max(candidates, key=lambda p: p.stat().st_mtime)
        shutil.copy(best, f"{OUTPUT_DIR}/{best.name}")
        print(f"✅ Checkpoint exporté : {best.name}")
    else:
        print(f"❌ Aucun checkpoint dans {SAVE_DIR}")

print()
print("=" * 60)
print("TERMINÉ — Modèle Phase 3 complet")
print("  → Onglet Output → télécharger le .pth")
print("  → Dashboard Recherche → charger le checkpoint → cribler des molécules")
print("="*60)